# Apple (AAPL) Stock Price — Time Series Forecasting Case Study

**Data**: `AAPL.csv` — NASDAQ daily OHLCV data, Aug 2018 – Aug 2019 (251 trading days), columns:
`Date, Open, High, Low, Close, Adj Close, Volume`.

**Agenda**
1. Exploratory Data Analysis (EDA)
2. ARIMA (no exogenous variables)
3. ARIMA with exogenous variables (ARIMAX / SARIMAX)
4. Facebook Prophet
5. Model comparison using RMSE

> Run `pip install statsmodels pmdarima prophet scikit-learn seaborn` first if any imports fail.


## 0. Setup — Import Libraries

In [ ]:
# Core
import numpy as np
import pandas as pd

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns
plt.rcParams['figure.figsize'] = (14, 6)
sns.set_style('darkgrid')

# Stats / ARIMA
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Optional: auto_arima (pip install pmdarima) — auto-selects (p,d,q)
try:
    from pmdarima import auto_arima
    HAVE_PMDARIMA = True
except ImportError:
    HAVE_PMDARIMA = False
    print("pmdarima not installed — run `pip install pmdarima` for auto order selection. "
          "Falling back to a manual order.")

# Prophet
try:
    from prophet import Prophet
except ImportError:
    from fbprophet import Prophet

# Metrics
from sklearn.metrics import mean_squared_error

import warnings
warnings.filterwarnings('ignore')


## 1. Load the Data

In [ ]:
DATA_PATH = "AAPL.csv"  # update this path if needed

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()


In [ ]:
# Parse dates, sort chronologically, set as index
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)
df = df.set_index('Date')

df.info()


## 2. Exploratory Data Analysis (EDA)

- Descriptive statistics & missing values
- Open/Close price and Volume trends
- Daily returns distribution
- Seasonal decomposition (trend / seasonality / residual)
- Stationarity check (Augmented Dickey-Fuller test)
- ACF / PACF plots to guide ARIMA order selection


In [ ]:
df.describe()


In [ ]:
df.isnull().sum()


In [ ]:
# Reindex to business-day frequency and forward-fill any gaps
# (NASDAQ holidays create small gaps in an otherwise daily series)
df = df.asfreq('B')
df = df.ffill()
df.isnull().sum()


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

axes[0].plot(df.index, df['Open'], label='Open', color='tab:blue')
axes[0].plot(df.index, df['Close'], label='Close', color='tab:orange')
axes[0].set_title('AAPL Open & Close Price (Aug 2018 – Aug 2019)')
axes[0].set_ylabel('Price (USD)')
axes[0].legend()

axes[1].plot(df.index, df['Volume'], color='tab:green')
axes[1].set_title('AAPL Trading Volume')
axes[1].set_ylabel('Volume')

plt.tight_layout()
plt.show()


In [ ]:
df['Daily_Return'] = df['Close'].pct_change()

plt.figure(figsize=(10, 5))
sns.histplot(df['Daily_Return'].dropna(), bins=50, kde=True)
plt.title('Distribution of AAPL Daily Returns')
plt.xlabel('Daily Return')
plt.show()


In [ ]:
decomposition = seasonal_decompose(df['Close'], model='additive', period=21)  # ~1 trading month

fig = decomposition.plot()
fig.set_size_inches(14, 10)
plt.tight_layout()
plt.show()


In [ ]:
def adf_test(series, title=''):
    result = adfuller(series.dropna())
    print(f'ADF Test: {title}')
    print(f'  ADF Statistic : {result[0]:.4f}')
    print(f'  p-value       : {result[1]:.4f}')
    for key, value in result[4].items():
        print(f'  Critical Value ({key}): {value:.4f}')
    if result[1] <= 0.05:
        print('  => Series is likely STATIONARY (reject H0)\n')
    else:
        print('  => Series is likely NON-STATIONARY (fail to reject H0)\n')

adf_test(df['Close'], 'Close Price (level)')
adf_test(df['Close'].diff(), 'Close Price (1st difference)')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4))
plot_acf(df['Close'].diff().dropna(), ax=axes[0], lags=40)
plot_pacf(df['Close'].diff().dropna(), ax=axes[1], lags=40)
axes[0].set_title('ACF — 1st Differenced Close')
axes[1].set_title('PACF — 1st Differenced Close')
plt.tight_layout()
plt.show()


## 3. Train / Test Split

Chronological split (no shuffling): train on the earlier portion, forecast/evaluate on the
later, held-out portion.


In [ ]:
TARGET = 'Close'
TEST_SIZE = 30  # last 30 trading days held out for testing

train = df.iloc[:-TEST_SIZE]
test = df.iloc[-TEST_SIZE:]

print(f'Train period : {train.index.min().date()} to {train.index.max().date()}  ({len(train)} rows)')
print(f'Test period  : {test.index.min().date()} to {test.index.max().date()}  ({len(test)} rows)')

plt.figure(figsize=(14, 5))
plt.plot(train.index, train[TARGET], label='Train')
plt.plot(test.index, test[TARGET], label='Test')
plt.title('Train / Test Split — AAPL Close Price')
plt.legend()
plt.show()


## 4. ARIMA Model (No Exogenous Variables)

Fit ARIMA(p, d, q) on the Close price using only its own history.


In [ ]:
if HAVE_PMDARIMA:
    auto_model = auto_arima(train[TARGET], seasonal=False, trace=True,
                             stepwise=True, suppress_warnings=True)
    order = auto_model.order
    print('Selected order (p,d,q):', order)
else:
    order = (1, 1, 1)  # manual fallback based on ADF/ACF/PACF analysis above
    print('Using manual fallback order (p,d,q):', order)


In [ ]:
arima_model = ARIMA(train[TARGET], order=order)
arima_fit = arima_model.fit()
print(arima_fit.summary())


In [ ]:
arima_forecast = arima_fit.forecast(steps=len(test))
arima_forecast.index = test.index

arima_rmse = np.sqrt(mean_squared_error(test[TARGET], arima_forecast))
print(f'ARIMA{order} RMSE on test set: {arima_rmse:.4f}')

plt.figure(figsize=(14, 5))
plt.plot(train.index[-90:], train[TARGET][-90:], label='Train (last 90d)')
plt.plot(test.index, test[TARGET], label='Actual (Test)')
plt.plot(test.index, arima_forecast, label='ARIMA Forecast', linestyle='--')
plt.title(f'ARIMA{order} Forecast vs Actual — RMSE = {arima_rmse:.4f}')
plt.legend()
plt.show()


## 5. ARIMA with Exogenous Variables (SARIMAX / ARIMAX)

Bring in exogenous regressors — `Open` and `Volume` — to see whether they improve on the
univariate ARIMA above.

**Note:** true out-of-sample forecasting requires exogenous variables to be forecast (or known)
in advance for the test period too. Here, as a benchmark, we use their actual historical values
for the test window — in production you'd forecast these independently.


In [ ]:
exog_cols = ['Open', 'Volume']

exog_train = train[exog_cols]
exog_test = test[exog_cols]

if HAVE_PMDARIMA:
    auto_exog_model = auto_arima(train[TARGET], exogenous=exog_train, seasonal=False,
                                  trace=True, stepwise=True, suppress_warnings=True)
    exog_order = auto_exog_model.order
    print('Selected order (p,d,q):', exog_order)
else:
    exog_order = order
    print('Using same fallback order (p,d,q):', exog_order)


In [ ]:
sarimax_model = SARIMAX(train[TARGET], exog=exog_train, order=exog_order,
                         enforce_stationarity=False, enforce_invertibility=False)
sarimax_fit = sarimax_model.fit(disp=False)
print(sarimax_fit.summary())


In [ ]:
sarimax_forecast = sarimax_fit.forecast(steps=len(test), exog=exog_test)
sarimax_forecast.index = test.index

sarimax_rmse = np.sqrt(mean_squared_error(test[TARGET], sarimax_forecast))
print(f'ARIMAX{exog_order} RMSE on test set: {sarimax_rmse:.4f}')

plt.figure(figsize=(14, 5))
plt.plot(train.index[-90:], train[TARGET][-90:], label='Train (last 90d)')
plt.plot(test.index, test[TARGET], label='Actual (Test)')
plt.plot(test.index, sarimax_forecast, label='ARIMAX Forecast', linestyle='--')
plt.title(f'ARIMAX{exog_order} Forecast vs Actual — RMSE = {sarimax_rmse:.4f}')
plt.legend()
plt.show()


## 6. Facebook Prophet

Prophet expects a dataframe with columns `ds` (date) and `y` (value). It automatically models
trend and weekly/yearly seasonality, and can incorporate holidays.


In [ ]:
prophet_train = train.reset_index()[['Date', TARGET]].rename(columns={'Date': 'ds', TARGET: 'y'})
prophet_test = test.reset_index()[['Date', TARGET]].rename(columns={'Date': 'ds', TARGET: 'y'})

prophet_model = Prophet(
    daily_seasonality=False,
    weekly_seasonality=True,
    yearly_seasonality=True,
    changepoint_prior_scale=0.05  # controls trend flexibility — tune as needed
)

# US market holidays (NASDAQ is closed on these days)
prophet_model.add_country_holidays(country_name='US')

prophet_model.fit(prophet_train)


In [ ]:
future = prophet_model.make_future_dataframe(periods=len(test), freq='B')
prophet_forecast_full = prophet_model.predict(future)

# Align forecast rows with the test period
prophet_forecast = prophet_forecast_full.set_index('ds').loc[prophet_test['ds'], 'yhat']

prophet_rmse = np.sqrt(mean_squared_error(prophet_test['y'].values, prophet_forecast.values))
print(f'Prophet RMSE on test set: {prophet_rmse:.4f}')


In [ ]:
fig1 = prophet_model.plot(prophet_forecast_full)
plt.title('Prophet Forecast — AAPL Close Price')
plt.show()

fig2 = prophet_model.plot_components(prophet_forecast_full)
plt.show()


In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(train.index[-90:], train[TARGET][-90:], label='Train (last 90d)')
plt.plot(test.index, test[TARGET], label='Actual (Test)')
plt.plot(test.index, prophet_forecast.values, label='Prophet Forecast', linestyle='--')
plt.title(f'Prophet Forecast vs Actual — RMSE = {prophet_rmse:.4f}')
plt.legend()
plt.show()


## 7. Model Comparison

In [ ]:
results = pd.DataFrame({
    'Model': [f'ARIMA{order}', f'ARIMAX{exog_order}', 'Prophet'],
    'RMSE': [arima_rmse, sarimax_rmse, prophet_rmse]
}).sort_values('RMSE').reset_index(drop=True)

print(results)

plt.figure(figsize=(8, 5))
sns.barplot(data=results, x='Model', y='RMSE', palette='viridis')
plt.title('RMSE Comparison Across Models')
plt.ylabel('RMSE (lower is better)')
plt.show()


In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(test.index, test[TARGET], label='Actual', linewidth=2, color='black')
plt.plot(test.index, arima_forecast, label=f'ARIMA{order}', linestyle='--')
plt.plot(test.index, sarimax_forecast, label=f'ARIMAX{exog_order}', linestyle='--')
plt.plot(test.index, prophet_forecast.values, label='Prophet', linestyle='--')
plt.title('All Model Forecasts vs Actual Close Price')
plt.legend()
plt.show()


## 8. Conclusion

- Compare which model achieved the lowest RMSE — Prophet's seasonality/holiday handling often
  helps on longer horizons, while ARIMAX can benefit from Volume/Open as leading indicators,
  and plain ARIMA is a solid, simple baseline.
- Caveat: ARIMAX above used the *actual* future Open/Volume values as exogenous inputs for
  benchmarking purposes — in a genuinely live forecast, these would need to be forecast
  independently or replaced with variables known in advance (e.g. calendar/holiday flags).
- Next steps: grid-search ARIMA orders and Prophet's `changepoint_prior_scale` /
  `seasonality_prior_scale`, add more exogenous regressors (sector ETF price, VIX), or
  benchmark against an LSTM.
